# Kepler Earth-like scoring (second dataset)
---
Source link - https://exoplanetarchive.ipac.caltech.edu/cgi-bin/TblView/nph-tblView?app=ExoTbls&config=PS

## Data Description

The dataset is obtained from the NASA Exoplanet Archive.

Each row represents a planetary system entry, including both planetary and stellar parameters.

Column definitions are based on the official documentation:
https://exoplanetarchive.ipac.caltech.edu/docs/API_PS_columns.html

In [461]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as st

In [462]:
data = pd.read_csv('../data/PS_2026.04.28_00.29.01.csv', comment='#')
data

,pl_name,hostname,default_flag,sy_snum,sy_pnum,discoverymethod,disc_year,disc_facility,soltype,pl_controv_flag,...,sy_vmagerr2,sy_kmag,sy_kmagerr1,sy_kmagerr2,sy_gaiamag,sy_gaiamagerr1,sy_gaiamagerr2,rowupdate,pl_pubdate,releasedate
0,11 Com b,11 Com,1,2,1,Radial Velocity,2007.0,Xinglong Station,Published Confirmed,0,...,-0.023,2.282,0.346,-0.346,4.44038,0.003848,-0.003848,2023-09-19,2023-08,2023-09-19
1,11 Com b,11 Com,0,2,1,Radial Velocity,2007.0,Xinglong Station,Published Confirmed,0,...,-0.023,2.282,0.346,-0.346,4.44038,0.003848,-0.003848,2014-05-14,2008-01,2014-05-14
2,11 Com b,11 Com,0,2,1,Radial Velocity,2007.0,Xinglong Station,Published Confirmed,0,...,-0.023,2.282,0.346,-0.346,4.44038,0.003848,-0.003848,2014-07-23,2011-08,2014-07-23
3,11 UMi b,11 UMi,0,1,1,Radial Velocity,2009.0,Thueringer Landessternwarte Tautenburg,Published Confirmed,0,...,-0.005,1.939,0.270,-0.270,4.56216,0.003903,-0.003903,2018-04-25,2011-08,2014-07-23
4,11 UMi b,11 UMi,1,1,1,Radial Velocity,2009.0,Thueringer Landessternwarte Tautenburg,Published Confirmed,0,...,-0.005,1.939,0.270,-0.270,4.56216,0.003903,-0.003903,2018-09-04,2017-03,2018-09-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39798,ups And d,ups And,0,2,3,Radial Velocity,1999.0,Multiple Observatories,Published Confirmed,0,...,-0.023,2.859,0.274,-0.274,3.98687,0.008937,-0.008937,2021-09-20,2021-07,2021-09-20
39799,ups Leo b,ups Leo,1,1,1,Radial Velocity,2021.0,Okayama Astrophysical Observatory,Published Confirmed,0,...,-0.023,2.184,0.248,-0.248,4.03040,0.008513,-0.008513,2022-01-10,2021-12,2022-01-10
39800,xi Aql b,xi Aql,0,1,1,Radial Velocity,2007.0,Okayama Astrophysical Observatory,Published Confirmed,0,...,-0.023,2.171,0.220,-0.220,4.42501,0.003837,-0.003837,2014-07-23,2011-08,2014-07-23
39801,xi Aql b,xi Aql,0,1,1,Radial Velocity,2007.0,Okayama Astrophysical Observatory,Published Confirmed,0,...,-0.023,2.171,0.220,-0.220,4.42501,0.003837,-0.003837,2014-05-14,2008-06,2014-05-14


In [463]:
data = data[data['soltype'] == 'Published Confirmed']

In [464]:
confirmed_planets = data.copy()

In [465]:
confirmed_planets = confirmed_planets[[
    'pl_name',
    'pl_rade',
    'pl_bmasse',
    'pl_orbsmax',
    'pl_eqt',
    'st_rad',
    'st_mass'
]]
confirmed_planets

,pl_name,pl_rade,pl_bmasse,pl_orbsmax,pl_eqt,st_rad,st_mass
0,11 Com b,NaN,4914.898486,1.178,NaN,13.760000,2.090000
1,11 Com b,NaN,6165.600000,1.290,NaN,19.000000,2.700000
2,11 Com b,NaN,5434.700000,1.210,NaN,NaN,2.600000
3,11 UMi b,NaN,3432.400000,1.510,NaN,NaN,1.700000
4,11 UMi b,NaN,4684.814200,1.530,NaN,29.790000,2.780000
...,...,...,...,...,...,...,...
39798,ups And d,NaN,1303.096469,2.517,NaN,1.622479,1.294197
39799,ups Leo b,NaN,162.092488,1.180,NaN,11.220000,1.480000
39800,xi Aql b,NaN,642.000000,0.580,NaN,NaN,1.400000
39801,xi Aql b,NaN,890.000000,0.680,NaN,12.000000,2.200000


In [466]:
confirmed_planets = confirmed_planets.rename(columns={
    'pl_name': 'planet_name',
    'pl_rade': 'planet_radius_earth',
    'pl_bmasse': 'planet_mass_earth',
    'pl_orbsmax': 'semi_major_axis_au',
    'pl_eqt': 'equilibrium_temp_k',
    'st_rad': 'star_radius_sun',
    'st_mass': 'star_mass_sun'
})

## Reference Values

All planetary and stellar features are compared to Earth and Sun reference values.

Since most features are already normalized (Earth = 1, Sun = 1), these constants allow direct similarity computation without additional scaling.

In [467]:
# Earth reference values
EARTH_RADIUS = 1        # Earth radii
EARTH_MASS = 1          # Earth masses
EARTH_DISTANCE = 1      # AU
EARTH_TEMP = 255        # Kelvin (approx)

# Sun reference values
SUN_RADIUS = 1          # Solar radii
SUN_MASS = 1            # Solar masses

In [468]:
def similarity(value, target):
    """
    Computes similarity score between a given value and a target reference.

    :param value: float or array-like
        The observed value of a planetary feature (radius, mass, temperature)

    :param target: float
        The reference Earth value for the corresponding feature

    :return: float or array-like
        A similarity score between 0 and 1, where:
        - 1 indicates identical to Earth
        - values closer to 0 indicate greater difference
    """
    return 1 / (1 + np.abs(value - target) / target)

## Model Differences

The two datasets use slightly different feature sets.

The first dataset includes star age as an additional factor, while the second dataset does not provide this information.

As a result, the scoring models are not identical, and the comparison reflects both:

- differences in the datasets
- differences in available features

This approach highlights how missing information can influence model results in real-world scenarios.

In [469]:
df2 = confirmed_planets.copy()

df2['radius_score'] = similarity(df2['planet_radius_earth'], EARTH_RADIUS)
df2['mass_score'] = similarity(df2['planet_mass_earth'], EARTH_MASS)
df2['distance_score'] = similarity(df2['semi_major_axis_au'], EARTH_DISTANCE)
df2['temp_score'] = similarity(df2['equilibrium_temp_k'], EARTH_TEMP)
df2['star_radius_score'] = similarity(df2['star_radius_sun'], SUN_RADIUS)
df2['star_mass_score'] = similarity(df2['star_mass_sun'], SUN_MASS)

In [470]:
# The weights are assigned based on domain knowledge,
# giving higher importance to planetary characteristics (mass, radius, temperature)
# compared to stellar properties.

weights_2 = {
    # Radius is critical for Earth-like classification (solid vs gas planets)
    'radius_score': 2,

    # Mass determines gravitational properties and atmospheric retention
    'mass_score': 2,

    # Orbital distance affects position in the habitable zone
    'distance_score': 1.5,

    # Temperature directly influences habitability conditions (liquid water)
    'temp_score': 2,

    # Star radius affects radiation output and planetary environment
    'star_radius_score': 1,

    # Star mass influences stellar lifetime and stability
    'star_mass_score': 1
}

In [471]:
# Calculate weighted Earth similarity score
# Each feature is multiplied by its importance (weight)
# Higher weights increase the influence of critical factors like radius, mass, and temperature

df2['total_score'] = (
    df2['radius_score'] * weights_2['radius_score'] +
    df2['mass_score'] * weights_2['mass_score'] +
    df2['distance_score'] * weights_2['distance_score'] +
    df2['temp_score'] * weights_2['temp_score'] +
    df2['star_radius_score'] * weights_2['star_radius_score'] +
    df2['star_mass_score'] * weights_2['star_mass_score']
) / sum(weights_2.values())  # Normalize by total weight to keep score between 0 and 1

# Round the final score for better readability
df2['total_score'] = df2['total_score'].round(2)

In [472]:
top_10 = df2.sort_values(by='total_score', ascending=False).head(9999)
top_10

,planet_name,planet_radius_earth,planet_mass_earth,semi_major_axis_au,equilibrium_temp_k,star_radius_sun,star_mass_sun,radius_score,mass_score,distance_score,temp_score,star_radius_score,star_mass_score,total_score
38030,TRAPPIST-1 e,0.918,0.62,0.02817,251.3,0.1170,0.0802,0.924214,0.724638,0.507143,0.985698,0.531067,0.520888,0.75
38035,TRAPPIST-1 f,1.045,0.68,0.03710,219.0,0.1170,0.0802,0.956938,0.757576,0.509450,0.876289,0.531067,0.520888,0.74
35352,LP 791-18 d,1.032,0.90,0.01992,395.5,0.1820,0.1390,0.968992,0.909091,0.505030,0.644753,0.550055,0.537346,0.73
38041,TRAPPIST-1 g,1.127,1.34,0.04510,198.6,0.1170,0.0802,0.887311,0.746269,0.511535,0.818882,0.531067,0.520888,0.71
19874,Kepler-20 f,0.952,1.40,0.13870,681.0,0.9164,0.9290,0.954198,0.714286,0.537259,0.374449,0.922850,0.933707,0.71
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15398,Kepler-1604 b,1.410,NaN,NaN,NaN,0.7600,0.8100,0.709220,NaN,NaN,NaN,0.806452,0.840336,NaN
15400,Kepler-1605 b,1.080,NaN,NaN,NaN,0.8200,0.8600,0.925926,NaN,NaN,NaN,0.847458,0.877193,NaN
15405,Kepler-1605 b,2.788,NaN,NaN,NaN,2.0640,NaN,0.358680,NaN,NaN,NaN,0.484496,NaN,NaN
15406,Kepler-1605 b,NaN,NaN,NaN,NaN,2.1319,NaN,NaN,NaN,NaN,NaN,0.469065,NaN,NaN


In [473]:
ds2 = top_10.copy()

## Loading Dataset 1

In this step, we load the processed dataset containing Earth-like planet scores from the first data source.

This dataset includes planets that have already been evaluated using the defined similarity-based scoring model.

The goal is to compare these scores with the results obtained from the second dataset, in order to analyze consistency, differences, and the impact of data source variations on the final ranking.

In [474]:
ds_1_earth_like_planet = pd.read_csv('../data/ds_1_earth_like_planets.csv')
ds_1_earth_like_planet

,planet_name,host_star,n_stars,n_planets,discovery_method,disc_year,disc_facility,orbital_period_days,planet_radius_earth,planet_mass_earth,...,star_type,orbital_period_cat,radius_score,mass_score,distance_score,temp_score,star_age_score,star_radius_score,star_mass_score,total_score
0,Kepler-1178 b,Kepler-1178,1,1,Transit,2016.0,Kepler,31.806340,1.070,1.240,...,K-type,Medium(10-100d),0.934579,0.806452,0.543360,0.674603,0.982906,0.800000,0.833333,0.79
1,Kepler-1417 b,Kepler-1417,1,1,Transit,2016.0,Kepler,20.350521,1.000,0.972,...,G-type(Sun-like),Medium(10-100d),1.000000,0.972763,0.537814,0.366906,0.896686,0.980392,0.980392,0.79
2,Kepler-1482 b,Kepler-1482,1,1,Transit,2016.0,Kepler,12.253832,1.010,1.010,...,G-type(Sun-like),Medium(10-100d),0.990099,0.990099,0.526227,0.376106,0.972516,0.862069,0.892857,0.78
3,Kepler-220 d,Kepler-220,1,4,Transit,2014.0,Kepler,28.122397,0.980,0.904,...,K-type,Medium(10-100d),0.980392,0.912409,0.544366,0.635910,0.766667,0.749625,0.786782,0.78
4,TOI-700 e,TOI-700,1,4,Transit,2023.0,Transiting Exoplanet Survey Satellite (TESS),27.809780,0.953,0.818,...,M-type(Red Dwarf),Medium(10-100d),0.955110,0.846024,0.535906,0.934408,0.597403,0.633312,0.630915,0.77
5,Kepler-438 b,Kepler-438,1,1,Transit,2015.0,Kepler,35.233190,1.120,1.460,...,K-type,Medium(10-100d),0.892857,0.684932,0.545256,0.885417,0.958333,0.675676,0.686813,0.77
6,TOI-700 d,TOI-700,1,4,Transit,2020.0,Transiting Exoplanet Survey Satellite (TESS),37.423960,1.073,1.250,...,M-type(Red Dwarf),Medium(10-100d),0.931966,0.800000,0.544455,0.948661,0.597403,0.633312,0.630915,0.77
7,Kepler-1464 c,Kepler-1464,1,2,Transit,2016.0,Kepler,5.327863,1.000,0.972,...,G-type(Sun-like),Short(1-10d),1.000000,0.972763,0.515732,0.249267,0.866290,0.990099,0.980392,0.77
8,Kepler-1328 b,Kepler-1328,1,1,Transit,2016.0,Kepler,4.521589,0.980,0.904,...,G-type(Sun-like),Short(1-10d),0.980392,0.912409,0.513347,0.210049,0.993521,0.990099,1.000000,0.76
9,Kepler-1605 b,Kepler-1605,1,1,Transit,2016.0,Kepler,85.756550,1.080,1.280,...,G-type(Sun-like),Medium(10-100d),0.925926,0.781250,0.611872,0.482955,0.982906,0.847458,0.877193,0.76


In [475]:
ds1 = ds_1_earth_like_planet[['planet_name', 'total_score']]
ds1

,planet_name,total_score
0,Kepler-1178 b,0.79
1,Kepler-1417 b,0.79
2,Kepler-1482 b,0.78
3,Kepler-220 d,0.78
4,TOI-700 e,0.77
5,Kepler-438 b,0.77
6,TOI-700 d,0.77
7,Kepler-1464 c,0.77
8,Kepler-1328 b,0.76
9,Kepler-1605 b,0.76


In [476]:
ds2 = ds2[ds2['planet_name'].isin(ds1['planet_name'])]
ds2

,planet_name,planet_radius_earth,planet_mass_earth,semi_major_axis_au,equilibrium_temp_k,star_radius_sun,star_mass_sun,radius_score,mass_score,distance_score,temp_score,star_radius_score,star_mass_score,total_score
9765,Kepler-1178 b,NaN,NaN,NaN,NaN,0.93646,0.85,NaN,NaN,NaN,NaN,0.940256,0.869565,NaN
9767,Kepler-1178 b,1.070,NaN,NaN,NaN,0.75000,0.80,0.934579,NaN,NaN,NaN,0.800000,0.833333,NaN
9770,Kepler-1178 b,1.379,NaN,NaN,NaN,1.14500,NaN,0.725163,NaN,NaN,NaN,0.873362,NaN,NaN
11832,Kepler-1328 b,0.980,NaN,NaN,NaN,0.99000,1.00,0.980392,NaN,NaN,NaN,0.990099,1.000000,NaN
11837,Kepler-1328 b,NaN,NaN,NaN,NaN,1.35747,1.00,NaN,NaN,NaN,NaN,0.736665,1.000000,NaN
11839,Kepler-1328 b,1.342,NaN,NaN,NaN,1.31600,NaN,0.745156,NaN,NaN,NaN,0.759878,NaN,NaN
13053,Kepler-1417 b,1.000,NaN,NaN,NaN,1.02000,1.02,1.000000,NaN,NaN,NaN,0.980392,0.980392,NaN
13054,Kepler-1417 b,1.592,NaN,NaN,NaN,1.57500,NaN,0.628141,NaN,NaN,NaN,0.634921,NaN,NaN
13059,Kepler-1417 b,NaN,NaN,NaN,NaN,1.58688,1.03,NaN,NaN,NaN,NaN,0.630167,0.970874,NaN
13635,Kepler-1464 c,1.000,NaN,NaN,NaN,1.01000,1.02,1.000000,NaN,NaN,NaN,0.990099,0.980392,NaN


## Conclusion

The comparison between the two datasets reveals significant differences in data completeness.

The second dataset contains fewer available features for many planets, which limits the applicability of the full scoring model used in the first dataset.

As a result, only a subset of planets can be directly compared, and the final scores differ noticeably between the two datasets.

These differences are not necessarily due to inconsistencies in the model, but rather reflect variations in data availability, measurement completeness, and dataset structure.

This highlights an important aspect of real-world data science: results are highly dependent on the quality and completeness of the data used.